# Classifier training — interactive walkthrough

Same code path as `src.train_classifier`, but exposed cell-by-cell so you can
tweak hyperparameters and inspect intermediate state without re-running the
whole pipeline. Use this for experimentation; `src.train_classifier` is the
reproducible CLI entrypoint.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import json
import pandas as pd
from src._classifier_dataset import IssueClassificationDataset
from src._classifier_model import build_model, count_trainable_params
from src._classifier_train import TrainConfig, train_classifier
from src._classifier_eval import evaluate

## 1. Load splits

In [ ]:
DATA = Path('../../data/raw/dataset')
train_df = pd.read_parquet(DATA / 'splits' / 'train.parquet')
val_df = pd.read_parquet(DATA / 'splits' / 'val.parquet')
test_df = pd.read_parquet(DATA / 'splits' / 'test.parquet')
manifest = json.loads((DATA / 'manifest.json').read_text())
print(f'train={len(train_df)} val={len(val_df)} test={len(test_df)}')
print(f'training_data_hash={manifest["raw_issues_sha256"][:12]}')

## 2. Build tokenized datasets

In [ ]:
cfg = TrainConfig()
train_ds = IssueClassificationDataset(train_df, max_seq_len=cfg.max_seq_len)
val_ds = IssueClassificationDataset(val_df, max_seq_len=cfg.max_seq_len)
test_ds = IssueClassificationDataset(test_df, max_seq_len=cfg.max_seq_len)
len(train_ds), len(val_ds), len(test_ds)

## 3. Build RoBERTa with freeze policy

We freeze embeddings + encoder layers 0..8 (out of 12) and train only the top
3 layers plus the classification head. Justified in DECISIONS.md.

In [ ]:
model = build_model(num_labels=4, freeze_through_layer=8)
print(f'trainable params: {count_trainable_params(model):,}')

## 4. Train (uses HF Trainer with early stopping on val macro-F1)

In [ ]:
OUT = Path('../../data/artifacts/classifier/roberta-issue-cls-v1')
model = train_classifier(model, train_ds, val_ds, cfg, OUT / '_hf')

## 5. Test-set evaluation

In [ ]:
report = evaluate(model, test_ds, batch_size=cfg.per_device_eval_batch_size)
print(f'accuracy={report.accuracy:.4f}  macro_f1={report.macro_f1:.4f}')
print(f'per-class F1: {report.per_class_f1}')
print(f'p50={report.p50_latency_ms:.1f}ms  p95={report.p95_latency_ms:.1f}ms')